# Feature Engineering

This notebook explores and develops feature transformations based on the findings from the Exploratory Data Analysis.

The objective is to design preprocessing steps that can be reused consistently during model training and inference, while preventing data leakage.

The final preprocessing logic will be implemented as reusable Python components and integrated into a scikit-learn pipeline.

## 1. Data Loading and Preparation

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent

RANDOM_STATE = 424242819

df = pd.read_csv(
    project_root / "data" / "raw" / "train.csv"
)

df = df.drop(columns=["Id"])

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1460, 80)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 2. Train/Test Split

The dataset is divided into training (80%) and test (20%) sets using a fixed random state to ensure reproducibility.

The test set will remain untouched during feature engineering, model selection, and hyperparameter optimization. All data-dependent preprocessing transformations will be fitted exclusively on the training data and subsequently applied to unseen observations.

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=RANDOM_STATE
)


print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (1168, 79)
X_test:  (292, 79)
y_train: (1168,)
y_test:  (292,)


## 3. Missing Value Imputation

Missing values will be handled using strategies appropriate for different feature types.

To prevent data leakage, imputation parameters will be learned exclusively from the training data and subsequently reused for validation, test, and unseen observations.

The following example demonstrates the `fit()` and `transform()` mechanism using a numerical feature.

In [3]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# Learn the median from training data
imputer.fit(X_train[["LotFrontage"]])

print("Learned median:", imputer.statistics_)

# Apply the learned median
lot_frontage_train = imputer.transform(X_train[["LotFrontage"]])

lot_frontage_test = imputer.transform(X_test[["LotFrontage"]])

print("Missing values in train:", 
      pd.isna(lot_frontage_train).sum())

print("Missing values in test:", 
      pd.isna(lot_frontage_test).sum())


Learned median: [70.]
Missing values in train: 0
Missing values in test: 0


### 3.1 Missing Values by Feature Type

Missing values are examined separately for numerical and categorical features to identify appropriate imputation strategies.

Some missing values represent unavailable information, while others indicate the absence of a particular property characteristic.

In [4]:
missing_summary = pd.DataFrame({
    "dtype": X_train.dtypes,
    "missing_count": X_train.isna().sum(),
    "missing_pct": X_train.isna().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)
missing_summary

,dtype,missing_count,missing_pct
PoolQC,object,1163,99.571918
MiscFeature,object,1122,96.061644
Alley,object,1098,94.006849
Fence,object,936,80.136986
MasVnrType,object,701,60.017123
FireplaceQu,object,546,46.746575
LotFrontage,float64,217,18.578767
GarageType,object,67,5.736301
GarageYrBlt,float64,67,5.736301
GarageFinish,object,67,5.736301


### 3.2 Missing Value Classification

Missing values can have different meanings depending on the feature.

For some variables, missing values indicate the absence of a property characteristic, such as a garage, basement, or alley access. These values can be represented by an explicit category such as "None".

For other variables, missing values represent unknown information and require statistical imputation or another appropriate strategy.

The classification below will guide the implementation of preprocessing transformations.

In [5]:
absence_features = [
    "Alley",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature"
]

absence_missing = missing_summary.loc[
    missing_summary.index.intersection(absence_features)
]

absence_missing

,dtype,missing_count,missing_pct
PoolQC,object,1163,99.571918
MiscFeature,object,1122,96.061644
Alley,object,1098,94.006849
Fence,object,936,80.136986
FireplaceQu,object,546,46.746575
GarageType,object,67,5.736301
GarageFinish,object,67,5.736301
GarageQual,object,67,5.736301
GarageCond,object,67,5.736301
BsmtExposure,object,30,2.568493


### 3.3 Remaining Missing Values

After identifying features where missing values may represent the absence of a property characteristic, the remaining features are separated into numerical and categorical groups.

Numerical features may require statistical imputation, while categorical features may require mode imputation or other feature-specific strategies.

The appropriate methods will be selected before implementing the preprocessing pipeline.

In [6]:
remaining_missing = missing_summary.drop(
    index=absence_features,
    errors="ignore"
)

numerical_missing = remaining_missing.loc[
    remaining_missing.index.intersection(
        X_train.select_dtypes(include="number").columns
    )
]

categorical_missing = remaining_missing.loc[
    remaining_missing.index.intersection(
        X_train.select_dtypes(include=["object", "category"]).columns
    )
]

print("Numerical features with missing values:")
display(numerical_missing)

print("Categorical features with missing values:")
display(categorical_missing)

Numerical features with missing values:


,dtype,missing_count,missing_pct
LotFrontage,float64,217,18.578767
GarageYrBlt,float64,67,5.736301
MasVnrArea,float64,7,0.599315


Categorical features with missing values:


,dtype,missing_count,missing_pct
MasVnrType,object,701,60.017123
Electrical,object,1,0.085616


### 3.4 Imputation Strategy Selection

Different imputation strategies will be applied depending on the meaning and data type of each feature.

- Numerical missing values will generally be imputed using the median.
- Missing values indicating the absence of a property characteristic will be represented explicitly.
- Remaining categorical missing values may be imputed using the most frequent category.
- Features with special dependencies will be examined individually before selecting an imputation strategy.

All learned imputation parameters will be fitted exclusively on training data within the preprocessing pipeline.

**GarageYrBlt**

In [7]:
garage_missing = X_train.loc[
    X_train["GarageYrBlt"].isna(),
    ["GarageType", "GarageArea", "GarageCars", "GarageYrBlt"]
]

print(f"Missing GarageYrBlt: {len(garage_missing)}")

garage_missing.value_counts(
    subset=["GarageType", "GarageArea", "GarageCars"],
    dropna=False
)

Missing GarageYrBlt: 67


GarageType  GarageArea  GarageCars
NaN         0           0             67
Name: count, dtype: int64

**GarageYrBlt – Missing Value Analysis**

All 67 observations with missing `GarageYrBlt` also have missing `GarageType`, `GarageArea = 0`, and `GarageCars = 0`.

This indicates that missing `GarageYrBlt` values correspond to properties without a garage rather than randomly missing construction years.

Therefore, missing values in `GarageYrBlt` will be represented using a dedicated absence value (`0`). This value will serve as a sentinel indicating that no garage is present, rather than an actual construction year.

**MasVnrType**

In [8]:
masonry_missing = X_train.loc[
    X_train["MasVnrType"].isna(),
    ["MasVnrType", "MasVnrArea"]
]

print(f"Missing MasVnrType: {len(masonry_missing)}")

masonry_missing.value_counts(
    subset=["MasVnrArea"],
    dropna=False
)

Missing MasVnrType: 701


MasVnrArea
0.0           690
NaN             7
1.0             2
312.0           1
344.0           1
Name: count, dtype: int64

**MasVnrType – Missing Value Analysis**

Among 701 observations with missing `MasVnrType`, 690 have `MasVnrArea = 0`, suggesting that the masonry veneer is absent.

However, 4 observations have a positive `MasVnrArea`, while 7 observations also have missing `MasVnrArea`.

Therefore, missing `MasVnrType` values should not be uniformly interpreted as the absence of masonry veneer.

A conditional imputation strategy will be considered:
- `"None"` when `MasVnrArea = 0`.
- `"Unknown"` when `MasVnrArea > 0` or `MasVnrArea` is missing.

This approach preserves the distinction between the absence of masonry veneer and unknown information.

In [9]:
area_missing = X_train.loc[
    X_train["MasVnrArea"].isna(),
    ["MasVnrType", "MasVnrArea"]
]

print(f"Missing MasVnrArea: {len(area_missing)}")

area_missing.value_counts(
    subset=["MasVnrType"],
    dropna=False
)

Missing MasVnrArea: 7


MasVnrType
NaN           7
Name: count, dtype: int64

**MasVnrArea – Summary**

All 7 observations with missing `MasVnrArea` also have missing `MasVnrType`.

Since the absence of masonry veneer cannot be confirmed for these observations, replacing missing areas with zero could incorrectly indicate that no veneer is present.

A preliminary strategy is to impute missing `MasVnrArea` values using the median of positive masonry veneer areas, while preserving the original missingness through an indicator.

The median will be learned exclusively from training data within the preprocessing pipeline.

The effectiveness of this strategy will be evaluated during model validation.

## 4. Missing Value Imputation Implementation

### 4.1 Absence-Related Categorical Features

Missing values in selected categorical features are treated as indicators of the absence of a property characteristic.

These values will be replaced with the explicit category "None" using `SimpleImputer`.

The transformation is first tested on training data without modifying the original dataset.

In [10]:
absence_imputer = SimpleImputer(
    strategy="constant",
    fill_value="None"
)

X_train_absence = X_train[absence_features].copy()

X_train_absence_imputed = absence_imputer.fit_transform(
    X_train_absence
)

X_train_absence_imputed = pd.DataFrame(
    X_train_absence_imputed,
    columns=absence_features,
    index=X_train.index
)

print("Missing values before:")
print(X_train_absence.isna().sum().sum())

print("\nMissing values after:")
print(X_train_absence_imputed.isna().sum().sum())

X_train_absence_imputed.head()

Missing values before:
5280

Missing values after:
0


,Alley,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PoolQC,Fence,MiscFeature
1356,None,TA,TA,No,Rec,Unf,None,Attchd,RFn,TA,TA,None,GdWo,None
989,None,Gd,Gd,No,Unf,Unf,None,Attchd,RFn,TA,TA,None,None,None
878,None,TA,TA,Av,ALQ,Unf,None,Detchd,Unf,TA,TA,None,MnPrv,Shed
1280,None,Gd,TA,No,GLQ,Unf,None,Attchd,RFn,TA,TA,None,None,None
184,None,Fa,TA,No,Unf,Unf,None,Attchd,Unf,TA,TA,None,MnPrv,None


### 4.2 Introduction to ColumnTransformer

ColumnTransformer allows different preprocessing operations to be applied to selected groups of features.

In this initial example, absence-related categorical features are imputed with the constant category "None", while missing values in LotFrontage are replaced with the median learned from training data.

The remaining features are preserved without modification.

In [11]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        # NEW TRANSFORMATION
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

X_train_transformed[
    ["Alley", "GarageType", "LotFrontage"]
].head()

,Alley,GarageType,LotFrontage
1356,None,Attchd,70.0
989,None,Attchd,65.0
878,None,Detchd,88.0
1280,None,Attchd,67.0
184,None,Attchd,92.0


In [12]:
print(X_train_transformed["LotFrontage"].isna().sum())
print(X_train_transformed["GarageType"].isna().sum())
print(X_train_transformed.shape)

0
0
(1168, 79)


### 4.3 Applying Preprocessing to Unseen Data

The fitted ColumnTransformer is applied to the holdout test set using `transform()`.

This ensures that the imputation parameters learned from the training data are reused without fitting the preprocessing steps on unseen observations.

In [13]:
# Apply the fitted preprocessor to the holdout test set
X_test_transformed = preprocessor.transform(X_test)

X_test_transformed = pd.DataFrame(
    X_test_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)

# Check the results
print("Test shape:", X_test_transformed.shape)

print(
    "Missing LotFrontage:",
    X_test_transformed["LotFrontage"].isna().sum()
)

print(
    "Missing GarageType:",
    X_test_transformed["GarageType"].isna().sum()
)

Test shape: (292, 79)
Missing LotFrontage: 0
Missing GarageType: 0


### 4.4 Numerical Imputation Strategy

Numerical features with missing values are reviewed before grouping them into preprocessing transformations.

Features requiring the same imputation strategy can be processed together using a single SimpleImputer within ColumnTransformer.

Features with special missing-value semantics will be handled separately.

In [14]:
numerical_missing_features = numerical_missing.index.tolist()

print("Numerical features with missing values:")
print(numerical_missing_features)

print("\nMissing values by feature:")
display(numerical_missing)

Numerical features with missing values:
['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

Missing values by feature:


,dtype,missing_count,missing_pct
LotFrontage,float64,217,18.578767
GarageYrBlt,float64,67,5.736301
MasVnrArea,float64,7,0.599315


### 4.5 Garage Year Built Imputation

Missing values in `GarageYrBlt` are associated with properties without a garage in the training dataset.

These missing values will be replaced with 0, which serves as a sentinel value indicating the absence of a garage rather than an actual construction year.

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        # NEW TRANSFORMATION
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

In [16]:
print("Missing GarageYrBlt before:",
      X_train["GarageYrBlt"].isna().sum())

print("Missing GarageYrBlt after:",
      X_train_transformed["GarageYrBlt"].isna().sum())

print("GarageYrBlt equal to 0:",
      (X_train_transformed["GarageYrBlt"] == 0).sum())

Missing GarageYrBlt before: 67
Missing GarageYrBlt after: 0
GarageYrBlt equal to 0: 67


### 4.6 Masonry Veneer Area Imputation

Missing values in `MasVnrArea` require special treatment because zero indicates the absence of masonry veneer, while missing values represent unknown information.

The median of positive masonry veneer areas will be examined as a potential imputation value. This statistic must be learned exclusively from training data.

In [17]:
mas_vnr_area = X_train["MasVnrArea"]

median_all = mas_vnr_area.median()

median_positive = mas_vnr_area.loc[
    mas_vnr_area > 0
].median()

print("Median of all non-missing values:", median_all)
print("Median of positive values:", median_positive)

print(
    "Number of positive values:",
    (mas_vnr_area > 0).sum()
)

print(
    "Number of zero values:",
    (mas_vnr_area == 0).sum()
)

Median of all non-missing values: 0.0
Median of positive values: 208.0
Number of positive values: 469
Number of zero values: 692


### 4.7 Custom Positive Median Imputer

A custom scikit-learn transformer is introduced to impute missing `MasVnrArea` values using the median of positive observations.

The imputation statistic is learned during `fit()` and reused during `transform()`, ensuring compatibility with cross-validation and production inference.

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin, OneToOneFeatureMixin

class PositiveMedianImputer(BaseEstimator, TransformerMixin, OneToOneFeatureMixin):

    def fit(self, X, y=None):
        values = np.asarray(X, dtype=float)

        positive_values = values[
            (values>0) & ~np.isnan(values)
        ]

        if len(positive_values)==0:
            raise ValueError("No positive values available.")

        self.median_ = np.median(positive_values)

        self.n_features_in_ = values.shape[1]
        
        return self

    def transform(self, X):
        values = np.asarray(X, dtype=float).copy()

        values[pd.isna(values)] = self.median_

        return values

In [19]:
positive_imputer = PositiveMedianImputer()

mas_vnr_train = positive_imputer.fit_transform(
    X_train[["MasVnrArea"]]
)

print("Learned median:", positive_imputer.median_)

print("Missing values before:",
      X_train["MasVnrArea"].isna().sum())

print("Missing values after:",
      np.isnan(mas_vnr_train).sum())

Learned median: 208.0
Missing values before: 7
Missing values after: 0


### 4.8 Integrating Custom Imputation into ColumnTransformer

The custom PositiveMedianImputer is integrated into ColumnTransformer to handle missing MasVnrArea values.

This allows the custom transformation to be fitted and applied alongside other preprocessing steps, ensuring consistent behavior during training, cross-validation, and inference.

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        # NEW TRANSFORMATION
        (
            "masonry_area_imputer",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

In [21]:
masonry_imputer = preprocessor.named_transformers_[
    "masonry_area_imputer"
]

print("Learned positive median:", masonry_imputer.median_)

print(
    "Missing MasVnrArea after:",
    X_train_transformed["MasVnrArea"].isna().sum()
)

print(
    "Imputed values:",
    X_train_transformed.loc[
        X_train["MasVnrArea"].isna(),
        "MasVnrArea"
    ].unique()
)

Learned positive median: 208.0
Missing MasVnrArea after: 0
Imputed values: [208.0]


### 4.9 Missing Value Indicator

A missing-value indicator is introduced for `MasVnrArea` to preserve information about originally missing observations.

The indicator equals 1 when the original value is missing and 0 otherwise. This allows the model to distinguish observed values from imputed values.

In [22]:
# Create an indicator based on original training data
mas_vnr_missing_indicator = (
    X_train["MasVnrArea"].isna().astype(int)
)

print(mas_vnr_missing_indicator.value_counts())

# Combine the imputed feature with the missing indicator
mas_vnr_preview = pd.DataFrame({
    "MasVnrArea": X_train_transformed["MasVnrArea"],
    "MasVnrArea_missing": mas_vnr_missing_indicator
    },
    index=X_train.index
)

# Display observations that originally had missing values
mas_vnr_preview.loc[
   mas_vnr_preview["MasVnrArea_missing"] == 1
]

MasVnrArea
0    1161
1       7
Name: count, dtype: int64


,MasVnrArea,MasVnrArea_missing
234,208.0,1
529,208.0,1
650,208.0,1
973,208.0,1
1278,208.0,1
936,208.0,1
977,208.0,1


### 4.10 Integrating Missing Indicators into Custom Imputation

The custom PositiveMedianImputer is extended to generate an additional missing-value indicator.

The transformer now returns both the imputed masonry veneer area and a binary feature indicating whether the original value was missing.

This preserves missingness information within the preprocessing pipeline.

In [23]:
from sklearn.utils.validation import check_is_fitted

class PositiveMedianImputer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        values =  np.asarray(X, dtype=float)

        if values.ndim != 2 or values.shape[1] != 1:
            raise ValueError("Expected exactly one input feature.")

        positive_values = values[
            (values > 0) & np.isfinite(values)
        ]


        if positive_values.size == 0:
            raise ValueError("No positive values available.")

        self.median_ = np.median(positive_values)
        self.n_features_in_ = 1

        return self

    def transform(self, X):
        check_is_fitted(self, "median_")

        values = np.asarray(X, dtype=float).copy()

        if values.ndim != 2 or values.shape[1] != 1:
            raise ValueError("Expected exactly one input feature.")

        missing_indicator = np.isnan(values).astype(int)

        values[np.isnan(values)] = self.median_

        return np.column_stack([
            values[:,0],
            missing_indicator[:,0]
        ])

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "median_")

        if input_features is None:
            feature_name = "MasVnrArea"
        else:
            feature_name = input_features[0]

        
        return np.array([
            feature_name,
            f"{feature_name}_missing"
        ], dtype=object)

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        # NEW TRANSFORMATION
        (
            "masonry_area_imputer",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

In [25]:
print("Shape:", X_train_transformed.shape)

print(
    X_train_transformed.loc[
        X_train["MasVnrArea"].isna(),
        ["MasVnrArea", "MasVnrArea_missing"]
    ]
)

Shape: (1168, 80)
     MasVnrArea MasVnrArea_missing
234       208.0                1.0
529       208.0                1.0
650       208.0                1.0
973       208.0                1.0
1278      208.0                1.0
936       208.0                1.0
977       208.0                1.0


### 4.11 Conditional Masonry Veneer Type Imputation

Missing values in `MasVnrType` are handled conditionally based on the original `MasVnrArea`.

When the masonry veneer area equals zero, the missing type is replaced with "None". Otherwise, missing types are replaced with "Unknown".

This rule preserves the distinction between the absence of masonry veneer and unavailable information.

In [26]:
masonry_preview = X_train[
    ["MasVnrType", "MasVnrArea"]
].copy()

# Identify missing veneer types
missing_type = masonry_preview["MasVnrType"].isna()

# Case 1: Missing type and zero veneer area
no_veneer = missing_type & (
    masonry_preview["MasVnrArea"] == 0
)

# Case 2: Missing type and positive or unknown area
unknown_veneer = missing_type & (
    (masonry_preview["MasVnrArea"] > 0)
    | masonry_preview["MasVnrArea"].isna()
)

# Apply conditional imputation
masonry_preview.loc[no_veneer, "MasVnrType"] = "None"
masonry_preview.loc[unknown_veneer, "MasVnrType"] = "Unknown"

# Check results
print("Assigned None:", no_veneer.sum())
print("Assigned Unknown:", unknown_veneer.sum())

print(
    "Remaining missing MasVnrType:",
    masonry_preview["MasVnrType"].isna().sum()
)

Assigned None: 690
Assigned Unknown: 11
Remaining missing MasVnrType: 0


### 4.12 Custom Conditional Masonry Type Imputer

A custom scikit-learn transformer is introduced to handle missing `MasVnrType` values based on the original `MasVnrArea`.

Missing masonry types are replaced with "None" when the area equals zero and "Unknown" otherwise.

The transformer preserves existing categories and can be integrated into ColumnTransformer and the model preprocessing pipeline.

In [27]:
class MasonryTypeImputer(TransformerMixin, BaseEstimator):

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Expected a pandas DataFrame.")

        required_columns = ["MasVnrType", "MasVnrArea"]

        if not all(col in X.columns for col in required_columns):
            raise ValueError("Required columns are missing.")

        self.n_features_in_ = 2
        self.feature_names_in_ = np.array(required_columns, dtype=object)

        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")

        data = X[self.feature_names_in_].copy()

        missing_type = data["MasVnrType"].isna()

        no_veneer = missing_type & (
            data["MasVnrArea"] == 0
        )

        data.loc[no_veneer, "MasVnrType"] = "None"

        data.loc[
            missing_type & ~no_veneer,
            "MasVnrType"
        ] = "Unknown"

        return data[["MasVnrType"]].to_numpy()

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")
        return np.array(["MasVnrType"], dtype=object)

In [28]:
masonry_type_imputer = MasonryTypeImputer()

masonry_type_train = masonry_type_imputer.fit_transform(
    X_train[["MasVnrType", "MasVnrArea"]]
)

masonry_type_train = pd.Series(
    masonry_type_train.ravel(),
    index=X_train.index,
    name="MasVnrType"
)

print("Remaining missing values:", masonry_type_train.isna().sum())

print(
    "Assigned None:",
    (masonry_type_train.loc[
        X_train["MasVnrType"].isna()
    ] == "None").sum()
)

print(
    "Assigned Unknown:",
    (masonry_type_train == "Unknown").sum()
)

Remaining missing values: 0
Assigned None: 690
Assigned Unknown: 11


### 4.13 Integrating Conditional Imputation into ColumnTransformer

The custom MasonryTypeImputer is integrated into ColumnTransformer alongside the existing preprocessing steps.

Both masonry-related transformers receive the original MasVnrArea values independently. The area transformer imputes missing numerical values and creates a missingness indicator, while the type transformer uses the original area to determine the appropriate categorical replacement.

This preserves the distinction between missing information and the absence of masonry veneer.

In [29]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area_imputer",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        # NEW TRANSFORMATION
        (
            "masonry_type_imputer",
            MasonryTypeImputer(),
            ["MasVnrType", "MasVnrArea"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)


In [30]:
print("Shape:", X_train_transformed.shape)

print(
    "Missing MasVnrType:",
    X_train_transformed["MasVnrType"].isna().sum()
)

print(
    "Missing MasVnrArea:",
    X_train_transformed["MasVnrArea"].isna().sum()
)

print(
    "Missing indicators:",
    X_train_transformed["MasVnrArea_missing"].sum()
)

print(
    "New None values:",
    (
        X_train_transformed.loc[
            X_train["MasVnrType"].isna(),
            "MasVnrType"
        ] == "None"
    ).sum()
)

print(
    "New Unknown values:",
    (
        X_train_transformed.loc[
            X_train["MasVnrType"].isna(),
            "MasVnrType"
        ] == "Unknown"
    ).sum()
)

Shape: (1168, 80)
Missing MasVnrType: 0
Missing MasVnrArea: 0
Missing indicators: 7.0
New None values: 690
New Unknown values: 11


### 4.14 Remaining Categorical Missing Values

Remaining categorical missing values are handled using the most frequent category.

The imputation strategy is learned exclusively from the training data and will be applied consistently during validation and inference.

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "absence_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area_imputer",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "masonry_type_imputer",
            MasonryTypeImputer(),
            ["MasVnrType", "MasVnrArea"]
        ),
        # NEW TRANSFORMATION
        (
            "categorical_mode_imputer",
            SimpleImputer(strategy="most_frequent"),
            ["Electrical"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

In [32]:
electrical_imputer = preprocessor.named_transformers_[
    "categorical_mode_imputer"
]

print(
    "Learned category:",
    electrical_imputer.statistics_[0]
)

print(
    "Missing before:",
    X_train["Electrical"].isna().sum()
)

print(
    "Missing after:",
    X_train_transformed["Electrical"].isna().sum()
)

print("Shape:", X_train_transformed.shape)

Learned category: SBrkr
Missing before: 1
Missing after: 0
Shape: (1168, 80)


### 4.15 Missing Values Audit

After applying the imputation strategies, the transformed training dataset is checked for remaining missing values.

This verification helps identify any features that still require preprocessing before categorical encoding and model training.

In [33]:
remaining_missing = (
    X_train_transformed
    .isna()
    .sum()
)

remaining_missing = remaining_missing[
    remaining_missing > 0 
].sort_values(ascending=False)

print("Total remaining missing values:")
print(X_train_transformed.isna().sum().sum())

print("\nFeatures with remaining missing values:")
display(remaining_missing)

Total remaining missing values:
0

Features with remaining missing values:


Series([], dtype: int64)

## 5. Categorical Feature Encoding

Categorical features require numerical representations before they can be used by the machine learning model.

The encoding strategy depends on the meaning of each feature. Nominal variables represent categories without a natural order, while ordinal variables describe ordered levels such as property quality or condition.

Before implementing encoding transformations, categorical features are identified and examined.

In [34]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_summary = pd.DataFrame({
    "n_categories": X_train_transformed[
        categorical_features
    ].nunique(),
    "categories": [
        X_train_transformed[col].unique().tolist()
        for col in categorical_features
    ]
}, index=categorical_features)

categorical_summary.sort_values(
    "n_categories",
    ascending=False
)

,n_categories,categories
Neighborhood,25,"[NAmes, Somerst, Sawyer, CollgCr, BrkSide, Edw..."
Exterior1st,15,"[HdBoard, VinylSd, AsbShng, CemntBd, MetalSd, ..."
Exterior2nd,15,"[HdBoard, VinylSd, Plywood, CmentBd, MetalSd, ..."
SaleType,9,"[WD, New, ConLD, COD, CWD, ConLI, ConLw, Oth, ..."
HouseStyle,8,"[1Story, 2Story, SFoyer, 1.5Fin, SLvl, 1.5Unf,..."
Condition1,8,"[Norm, RRAn, Feedr, Artery, PosA, RRAe, PosN, ..."
BsmtFinType1,7,"[Rec, Unf, ALQ, GLQ, BLQ, LwQ, None]"
RoofMatl,7,"[CompShg, Tar&Grv, WdShake, WdShngl, Metal, Ro..."
GarageType,7,"[Attchd, Detchd, None, BuiltIn, CarPort, Basme..."
BsmtFinType2,7,"[Unf, BLQ, Rec, LwQ, None, GLQ, ALQ]"


### 5.1 Ordinal and Nominal Feature Classification

Categorical features are divided into ordinal and nominal groups based on their semantic meaning.

Ordinal variables represent ordered levels of quality, condition, or functionality. Nominal variables represent categories without a meaningful ranking.

This classification will determine which encoding transformations are applied in the preprocessing pipeline.

In [35]:
ordinal_features = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Functional",
    "PavedDrive",
    "Utilities",
    "LandSlope",
    "LotShape"
]

nominal_features = [
    col for col in categorical_features
    if col not in ordinal_features
]

print("Ordinal features:", len(ordinal_features))
print("Nominal features:", len(nominal_features))

print("\nOrdinal variables:")
print(ordinal_features)

print("\nNominal variables:")
print(nominal_features)

Ordinal features: 19
Nominal features: 24

Ordinal variables:
['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Functional', 'PavedDrive', 'Utilities', 'LandSlope', 'LotShape']

Nominal variables:
['MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


### 5.2 Ordinal Category Mapping

Ordinal categorical variables are encoded according to their natural ordering.

Quality and condition features generally follow the hierarchy:
Poor < Fair < Typical < Good < Excellent.

For features where missing values indicate the absence of a property characteristic, the explicit "None" category is placed below the lowest quality level.

The category mappings will be passed to OrdinalEncoder within the preprocessing pipeline.

In [36]:
quality_order = [
    "None",
    "Po",
    "Fa",
    "TA",
    "Gd",
    "Ex"
]

quality_features = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC"
]

ordinal_categories = {
    feature: quality_order
    for feature in quality_features
}

ordinal_categories

{'ExterQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'ExterCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'HeatingQC': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'KitchenQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 'PoolQC': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']}

### 5.3 Additional Ordinal Category Mappings

The remaining ordinal features describe different property characteristics, such as basement exposure, garage finishing, functionality, and lot configuration.

Each feature requires an individual category order based on its semantic meaning.

These mappings will be used to configure OrdinalEncoder in the preprocessing pipeline.

In [37]:
additional_ordinal_categories = {
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],

    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

    "GarageFinish": ["None", "Unf", "RFn", "Fin"],

    "Functional": [
        "Sal", "Sev", "Maj2", "Maj1",
        "Mod", "Min2", "Min1", "Typ"
    ],

    "PavedDrive": ["N", "P", "Y"],

    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],

    "LandSlope": ["Sev", "Mod", "Gtl"],

    "LotShape": ["IR3", "IR2", "IR1", "Reg"]
}

ordinal_categories.update(additional_ordinal_categories)

print("Number of mapped ordinal features:", len(ordinal_categories))

Number of mapped ordinal features: 19


In [38]:
unmapped_features = set(ordinal_features) - set(ordinal_categories)

print("Unmapped ordinal features:", unmapped_features)

Unmapped ordinal features: set()


### 5.4 Ordinal Category Validation

Before implementing OrdinalEncoder, the predefined category mappings are validated against the categories observed in the training dataset.

This step identifies categories that are present in the data but missing from the mappings, helping prevent encoding errors and ensuring consistency.

In [39]:
unrecognized_categories = {}

for feature in ordinal_features:

    observed = set(
        X_train_transformed[feature].dropna().unique()
    )

    expected = set(ordinal_categories[feature])

    unknown = observed - expected

    if unknown:
        unrecognized_categories[feature] = unknown

print("Unrecognized categories:")
print(unrecognized_categories)

Unrecognized categories:
{}


### 5.5 Ordinal Encoding Implementation

Ordinal categorical features are encoded using predefined category hierarchies.

The OrdinalEncoder converts categories into numerical values while preserving their specified ordering.

Unknown categories encountered during inference will be encoded as -1 to prevent transformation errors.

In [40]:
from sklearn.preprocessing import OrdinalEncoder

# Preserve the order of features
ordinal_category_lists = [
    ordinal_categories[feature]
    for feature in ordinal_features
]

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_category_lists,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [41]:
# Fit the encoder on the imputed training data
X_train_ordinal = ordinal_encoder.fit_transform(
    X_train_transformed[ordinal_features]
)

X_train_ordinal = pd.DataFrame(
    X_train_ordinal,
    columns=ordinal_features,
    index=X_train.index
)

X_train_ordinal.head()

,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,HeatingQC,KitchenQual,FireplaceQu,GarageFinish,GarageQual,GarageCond,PoolQC,Functional,PavedDrive,Utilities,LandSlope,LotShape
1356,3.0,3.0,3.0,3.0,1.0,3.0,1.0,3.0,3.0,0.0,2.0,3.0,3.0,0.0,7.0,2.0,3.0,2.0,3.0
989,4.0,3.0,4.0,4.0,1.0,1.0,1.0,5.0,4.0,0.0,2.0,3.0,3.0,0.0,7.0,2.0,3.0,2.0,3.0
878,3.0,3.0,3.0,3.0,3.0,5.0,1.0,3.0,4.0,0.0,1.0,3.0,3.0,0.0,5.0,2.0,3.0,2.0,2.0
1280,4.0,3.0,4.0,3.0,1.0,6.0,1.0,5.0,4.0,0.0,2.0,3.0,3.0,0.0,7.0,2.0,3.0,2.0,2.0
184,3.0,3.0,2.0,3.0,1.0,1.0,1.0,4.0,3.0,0.0,1.0,3.0,3.0,0.0,7.0,2.0,3.0,2.0,2.0


In [42]:
print("Original KitchenQual:")
print(X_train_transformed["KitchenQual"].value_counts())

print("\nEncoded KitchenQual:")
print(X_train_ordinal["KitchenQual"].value_counts())

print("\nShape:", X_train_ordinal.shape)

Original KitchenQual:
KitchenQual
TA    603
Gd    458
Ex     79
Fa     28
Name: count, dtype: int64

Encoded KitchenQual:
KitchenQual
3.0    603
4.0    458
5.0     79
2.0     28
Name: count, dtype: int64

Shape: (1168, 19)


### 5.6 Combining Imputation and Ordinal Encoding

A scikit-learn Pipeline is used to apply missing-value imputation before ordinal encoding.

The imputation transformer is fitted on the training data, and the resulting categorical values are passed to OrdinalEncoder.

This ensures that both preprocessing steps are consistently applied during training and inference.

In [43]:
from sklearn.pipeline import Pipeline

kitchen_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OrdinalEncoder(
            categories=[ordinal_categories["KitchenQual"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

kitchen_encoded = kitchen_pipeline.fit_transform(
    X_train[["KitchenQual"]]
)

print("Pipeline output shape:", kitchen_encoded.shape)
print("First 10 encoded values:", kitchen_encoded[:10].ravel())
print("Missing values:", pd.isna(kitchen_encoded).sum())

Pipeline output shape: (1168, 1)
First 10 encoded values: [3. 4. 4. 4. 3. 3. 3. 3. 4. 3.]
Missing values: 0


### 5.7 Ordinal Encoding Pipelines

Ordinal features are divided into groups according to their missing-value semantics.

Features describing the quality or condition of existing property elements are imputed using the most frequent category when necessary.

Features where missing values indicate the absence of a property characteristic are imputed using the explicit category "None".

Both groups are subsequently encoded using their predefined ordinal category mappings.

In [44]:
ordinal_absence_features = [
    feature for feature in ordinal_features
    if feature in absence_features
]

ordinal_other_features = [
    feature for feature in ordinal_features
    if feature not in absence_features
]

ordinal_absence_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="None"
        )
    ),
    (
        "encoder",
        OrdinalEncoder(
            categories=[
                ordinal_categories[feature]
                for feature in ordinal_absence_features
            ],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

ordinal_other_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OrdinalEncoder(
            categories=[
                ordinal_categories[feature]
                for feature in ordinal_other_features
            ],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

print("Ordinal absence features:", ordinal_absence_features)
print("Other ordinal features:", ordinal_other_features)

Ordinal absence features: ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC']
Other ordinal features: ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual', 'Functional', 'PavedDrive', 'Utilities', 'LandSlope', 'LotShape']


In [45]:
nominal_absence_features = [
    feature for feature in absence_features
    if feature not in ordinal_features
]

print("Nominal absence features:")
print(nominal_absence_features)

print("\nNumber of features:", len(nominal_absence_features))

print(
    "\nOverlap with ordinal features:",
    set(nominal_absence_features) & set(ordinal_features)
)

Nominal absence features:
['Alley', 'GarageType', 'Fence', 'MiscFeature']

Number of features: 4

Overlap with ordinal features: set()


### 5.8 Integrating Ordinal Encoding into ColumnTransformer

The ordinal encoding pipelines are integrated into the main preprocessing transformer.

Ordinal features with absence-related missing values are imputed using "None", while the remaining ordinal features use the most frequent category when necessary.

Each group is subsequently encoded using predefined category orders. This avoids duplicated features and keeps imputation and encoding within the same preprocessing workflow.

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            SimpleImputer(
                strategy="constant",
                fill_value="None"
            ),
            nominal_absence_features
        ),
        (
            "lot_frontage_imputer",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year_imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area_imputer",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "masonry_type_imputer",
            MasonryTypeImputer(),
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "categorical_mode_imputer",
            SimpleImputer(strategy="most_frequent"),
            ["Electrical"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)


In [47]:
print("Shape:", X_train_transformed.shape)

print(
    "Total missing values:",
    X_train_transformed.isna().sum().sum()
)

print(
    "\nEncoded KitchenQual:",
    X_train_transformed["KitchenQual"].unique()
)

print(
    "\nEncoded GarageQual:",
    X_train_transformed["GarageQual"].unique()
)

print(
    "\nDuplicated columns:",
    X_train_transformed.columns.duplicated().sum()
)

Shape: (1168, 80)
Total missing values: 0

Encoded KitchenQual: [3.0 4.0 5.0 2.0]

Encoded GarageQual: [3.0 0.0 2.0 4.0 1.0 5.0]

Duplicated columns: 0


### 5.9 One-Hot Encoding for Nominal Features

Nominal features represent categories without a natural ordering.

OneHotEncoder converts each category into a separate binary feature, avoiding the introduction of an artificial numerical hierarchy.

The encoder is first tested on GarageType before being integrated into the main preprocessing pipeline.

In [48]:
from sklearn.preprocessing import OneHotEncoder

garage_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

garage_encoded = garage_encoder.fit_transform(
    X_train_transformed[["GarageType"]]
)

garage_encoded = pd.DataFrame(
    garage_encoded,
    columns=garage_encoder.get_feature_names_out(
        ["GarageType"]
    ),
    index=X_train.index
)

print("Original shape:", X_train[["GarageType"]].shape)
print("Encoded shape:", garage_encoded.shape)

garage_encoded.head()


Original shape: (1168, 1)
Encoded shape: (1168, 7)


,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None
1356,0.0,1.0,0.0,0.0,0.0,0.0,0.0
989,0.0,1.0,0.0,0.0,0.0,0.0,0.0
878,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1280,0.0,1.0,0.0,0.0,0.0,0.0,0.0
184,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [49]:
print("Learned categories:")
print(garage_encoder.categories_[0])

print("\nEncoded columns:")
print(garage_encoded.columns.tolist())

Learned categories:
['2Types' 'Attchd' 'Basment' 'BuiltIn' 'CarPort' 'Detchd' 'None']

Encoded columns:
['GarageType_2Types', 'GarageType_Attchd', 'GarageType_Basment', 'GarageType_BuiltIn', 'GarageType_CarPort', 'GarageType_Detchd', 'GarageType_None']


### 5.10 Preparing Nominal Features for Encoding

Nominal categorical features will be encoded using OneHotEncoder.

Although MSSubClass is stored as a numerical variable, it represents building classifications rather than continuous numerical values. Therefore, it will also be treated as a nominal categorical feature.

The complete set of nominal features is prepared before integrating encoding into the preprocessing pipeline.

In [50]:
# Include MSSubClass as a nominal feature
nominal_features_all = nominal_features + ["MSSubClass"]

print("Number of nominal features:", len(nominal_features_all))

print("\nNominal features:")
print(nominal_features_all)

print(
    "\nOverlapping ordinal and nominal features:",
    set(ordinal_features) & set(nominal_features_all)
)

Number of nominal features: 25

Nominal features:
['MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition', 'MSSubClass']

Overlapping ordinal and nominal features: set()


### 5.11 Nominal Encoding Pipelines

Nominal features are grouped according to their missing-value handling requirements.

Features where missing values indicate the absence of a property characteristic are imputed with "None" before One-Hot Encoding.

Other nominal features are encoded using a separate pipeline, while MasVnrType is handled by its dedicated conditional imputer.

This separation preserves the intended imputation strategies and prevents inconsistent preprocessing.

In [51]:
# Nominal features with absence-related missing values
nominal_absence_features = [
    feature for feature in nominal_features_all
    if feature in absence_features
]

# Other nominal features, excluding MasVnrType
nominal_other_features = [
    feature for feature in nominal_features_all
    if feature not in nominal_absence_features
    and feature != "MasVnrType"
]

# Pipeline for absence-related nominal features
nominal_absence_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="None"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# Pipeline for other nominal features
nominal_other_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

print("Nominal absence features:", nominal_absence_features)
print("Other nominal features:", nominal_other_features)

print(
    "\nTotal nominal features:",
    len(nominal_absence_features)
    + len(nominal_other_features)
    + 1  # MasVnrType
)


Nominal absence features: ['Alley', 'GarageType', 'Fence', 'MiscFeature']
Other nominal features: ['MSZoning', 'Street', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'SaleType', 'SaleCondition', 'MSSubClass']

Total nominal features: 25


### 5.12 Conditional Masonry Type Encoding

A dedicated pipeline combines conditional imputation of MasVnrType with One-Hot Encoding.

MasonryTypeImputer uses the original MasVnrArea to distinguish the absence of masonry veneer from unknown veneer types.

The resulting categorical feature is then encoded using OneHotEncoder, ensuring consistent preprocessing during training and inference.

In [52]:
masonry_type_pipeline = Pipeline([
    (
        "imptuer",
        MasonryTypeImputer()
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

In [53]:
masonry_type_encoded = masonry_type_pipeline.fit_transform(
    X_train[["MasVnrType", "MasVnrArea"]]
)

masonry_type_columns = (
    masonry_type_pipeline.get_feature_names_out()
)

masonry_type_encoded = pd.DataFrame(
    masonry_type_encoded,
    columns=masonry_type_columns,
    index=X_train.index
)

print("Shape:", masonry_type_encoded.shape)

print("\nEncoded columns:")
print(masonry_type_encoded.columns.tolist())

print("\nMissing values:")
print(masonry_type_encoded.isna().sum().sum())

masonry_type_encoded.head()

Shape: (1168, 5)

Encoded columns:
['MasVnrType_BrkCmn', 'MasVnrType_BrkFace', 'MasVnrType_None', 'MasVnrType_Stone', 'MasVnrType_Unknown']

Missing values:
0


,MasVnrType_BrkCmn,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,MasVnrType_Unknown
1356,0.0,1.0,0.0,0.0,0.0
989,0.0,0.0,1.0,0.0,0.0
878,0.0,0.0,1.0,0.0,0.0
1280,0.0,1.0,0.0,0.0,0.0
184,0.0,0.0,1.0,0.0,0.0


### 5.13 Integrating Nominal Encoding into ColumnTransformer

Nominal encoding pipelines are integrated into the main preprocessing transformer alongside ordinal encoding and numerical imputation.

Each categorical feature is assigned to an appropriate encoding pipeline, while special numerical transformations are preserved.

The resulting dataset contains numerical representations of all categorical features and can be used in subsequent feature engineering and model training stages.

In [54]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            nominal_absence_pipeline,
            nominal_absence_features
        ),
        (
            "nominal_other",
            nominal_other_pipeline,
            nominal_other_features
        ),
        (
            "masonry_type",
            masonry_type_pipeline,
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "lot_frontage",
            SimpleImputer(strategy="median"),
            ["LotFrontage"]
        ),
        (
            "garage_year",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)


In [55]:
print("Transformed shape:", X_train_transformed.shape)

print(
    "Total missing values:",
    X_train_transformed.isna().sum().sum()
)

print(
    "Duplicated columns:",
    X_train_transformed.columns.duplicated().sum()
)

print(
    "Object columns remaining:",
    X_train_transformed.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()
)

Transformed shape: (1168, 240)
Total missing values: 0
Duplicated columns: 0
Object columns remaining: []


### 5.14 Preprocessing Output Validation

The transformed training dataset is validated to ensure that all features are suitable for machine learning model training.

The validation checks for missing values, duplicated feature names, non-numeric values, and the overall dimensionality of the transformed dataset.

These checks help identify potential preprocessing issues before moving to numerical feature engineering.

In [56]:
# 1. Dataset dimensions
print("Original shape:", X_train.shape)
print("Transformed shape:", X_train_transformed.shape)

# 2. Missing values
print(
    "\nTotal missing values:",
    X_train_transformed.isna().sum().sum()
)

# 3. Duplicated column names
print(
    "\nDuplicated columns:",
    X_train_transformed.columns.duplicated().sum()
)

# 4. Verify numeric compatibility
X_train_numeric = X_train_transformed.apply(
    pd.to_numeric,
    errors="coerce"
)

conversion_failures = (
    X_train_numeric.isna()
    & X_train_transformed.notna()
).sum()

conversion_failures = conversion_failures[
    conversion_failures > 0
]

print("\nColumns with non-numeric values:")
print(conversion_failures)

# 5. Final numeric types
print("\nData types after conversion:")
print(X_train_numeric.dtypes.value_counts())

Original shape: (1168, 79)
Transformed shape: (1168, 240)

Total missing values: 0

Duplicated columns: 0

Columns with non-numeric values:
Series([], dtype: int64)

Data types after conversion:
float64    240
Name: count, dtype: int64


### 5.15 Preprocessing Validation on Unseen Categories

The fitted preprocessing transformer is tested on a small sample containing a previously unseen categorical value.

The objective is to verify that new categories can be handled without refitting the preprocessing pipeline and that the output feature structure remains consistent.

In [57]:

# Create a small sample of observations
X_new = X_train.iloc[:3].copy()

# Introduce an unseen nominal category
X_new.loc[X_new.index[0], "Neighborhood"] = "NewNeighborhood"

# Apply the previously fitted preprocessor
X_new_transformed = preprocessor.transform(X_new)

# Convert to DataFrame for inspection
X_new_transformed = pd.DataFrame(
    X_new_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_new.index
)

print("New data shape:", X_new_transformed.shape)

print(
    "Same feature names:",
    X_new_transformed.columns.equals(
        X_train_transformed.columns
    )
)

print(
    "Missing values:",
    X_new_transformed.isna().sum().sum()
)

# Check how the unseen category was encoded
neighborhood_columns = [
    col for col in X_new_transformed.columns
    if col.startswith("Neighborhood_")
]

print(
    "Encoded values for unseen Neighborhood:",
    X_new_transformed.loc[
        X_new.index[0],
        neighborhood_columns
    ].tolist()
)


New data shape: (3, 240)
Same feature names: True
Missing values: 0
Encoded values for unseen Neighborhood: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


### 5.16 Final Preprocessing Validation

The preprocessing output is checked for numerical compatibility, missing values, and infinite values.

These checks verify whether the transformed features can be passed directly to a machine learning model without additional manual conversions.

In [58]:

# Inspect the actual output of ColumnTransformer
X_output = preprocessor.transform(X_train.iloc[:5])

print("Output type:", type(X_output))
print("Output dtype:", X_output.dtype)
print("Output shape:", X_output.shape)

# Validate numeric compatibility
try:
    X_numeric = np.asarray(X_output, dtype=float)

    print("\nNumeric conversion: successful")
    print("Missing values:", np.isnan(X_numeric).sum())
    print("Infinite values:", np.isinf(X_numeric).sum())
    print("All values finite:", np.isfinite(X_numeric).all())

except (ValueError, TypeError) as error:
    print("\nNumeric conversion failed:")
    print(error)


Output type: <class 'numpy.ndarray'>
Output dtype: float64
Output shape: (5, 240)

Numeric conversion: successful
Missing values: 0
Infinite values: 0
All values finite: True


## 6. Numerical Feature Engineering

Numerical feature engineering aims to improve the representation of continuous variables and create additional features that may provide useful predictive information.

Based on the exploratory data analysis, selected right-skewed numerical features are considered for logarithmic transformation.

The proposed transformations will be implemented within the preprocessing pipeline and evaluated during model validation.

In [59]:
log_features = [
    "LotArea",
    "LotFrontage",
    "1stFlrSF",
    "GrLivArea"
]

print(
    X_train[log_features].describe()
    .loc[["min", "25%", "50%", "75%", "max"]]
)

      LotArea  LotFrontage  1stFlrSF  GrLivArea
min    1300.0         21.0    334.00     334.00
25%    7700.0         60.0    885.75    1120.75
50%    9552.0         70.0   1091.50    1461.50
75%   11695.5         80.5   1392.50    1788.00
max  215245.0        182.0   3228.00    4676.00


### 6.1 Logarithmic Transformation

Selected numerical features exhibit right-skewed distributions and are candidates for logarithmic transformation.

The log1p transformation is applied using FunctionTransformer to ensure compatibility with the preprocessing pipeline.

The transformation is first demonstrated on LotArea before being integrated into the main preprocessing workflow.

In [60]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(
    np.log1p,
    inverse_func=np.expm1,
    feature_names_out="one-to-one"
)

lot_area_log = log_transformer.fit_transform(X_train[["LotArea"]])

comparison = pd.DataFrame({
    "LotArea_original": X_train["LotArea"],
    "LotArea_log": lot_area_log["LotArea"]
})

comparison.head()

,LotArea_original,LotArea_log
1356,9477,9.156729
989,8125,9.002824
878,11782,9.374413
1280,9808,9.191056
184,7438,8.914492


In [61]:
log_area_restored = log_transformer.inverse_transform(lot_area_log)

print(
    "Original and restored values equal:",
    np.allclose(
        X_train[["LotArea"]].to_numpy(),
        log_area_restored.to_numpy()
    )
)

Original and restored values equal: True


### 6.2 Log Transformation with Missing Value Imputation

The LotFrontage feature contains missing values and therefore requires imputation before logarithmic transformation.

A Pipeline combines median imputation with the log1p transformation, ensuring that the imputation statistic is learned during training and reused during inference.

In [62]:
lot_frontage_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "log_transformer",
        FunctionTransformer(
            np.log1p,
            inverse_func=np.expm1,
            feature_names_out="one-to-one"
        )
    )
])

lot_frontage_log = lot_frontage_pipeline.fit_transform(X_train[["LotFrontage"]])

print("Shape:", lot_frontage_log.shape)
print("Missing values:", np.isnan(lot_frontage_log).sum())

print(
    "Median learned:",
    lot_frontage_pipeline.named_steps["imputer"].statistics_[0]
)

Shape: (1168, 1)
Missing values: 0
Median learned: 70.0


In [63]:
lot_frontage_pipeline.named_steps

{'imputer': SimpleImputer(strategy='median'),
 'log_transformer': FunctionTransformer(feature_names_out='one-to-one', func=<ufunc 'log1p'>,
                     inverse_func=<ufunc 'expm1'>)}

### 6.3 Integrating Log Transformations into ColumnTransformer

Logarithmic transformations are integrated into the main preprocessing workflow.

Features without missing values are transformed directly using log1p, while LotFrontage is first imputed using the median learned from training data.

All transformations remain part of the preprocessing pipeline to ensure consistency during cross-validation and inference.

In [64]:
log_features_no_missing = [
    "LotArea",
    "1stFlrSF",
    "GrLivArea"
]

log_transformer = FunctionTransformer(
    np.log1p,
    inverse_func=np.expm1,
    feature_names_out="one-to-one"
)

In [65]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            nominal_absence_pipeline,
            nominal_absence_features
        ),
        (
            "nominal_other",
            nominal_other_pipeline,
            nominal_other_features
        ),
        (
            "masonry_type",
            masonry_type_pipeline,
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "lot_frontage",
            lot_frontage_pipeline,
            ["LotFrontage"]
        ),
        (
            "garage_year",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "log_features",
            log_transformer,
            log_features_no_missing
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)


In [66]:
print("Transformed shape:", X_train_transformed.shape)

print(
    "Total missing values:",
    X_train_transformed.isna().sum().sum()
)

print(
    "Duplicated columns:",
    X_train_transformed.columns.duplicated().sum()
)

print("\nOriginal LotArea:")
print(X_train["LotArea"].head())

print("\nTransformed LotArea:")
print(X_train_transformed["LotArea"].head())

print(
    "\nLog transformation correct:",
    np.allclose(
        X_train_transformed["LotArea"].astype(float),
        np.log1p(X_train["LotArea"])
    )
)

Transformed shape: (1168, 240)
Total missing values: 0
Duplicated columns: 0

Original LotArea:
1356     9477
989      8125
878     11782
1280     9808
184      7438
Name: LotArea, dtype: int64

Transformed LotArea:
1356    9.156729
989     9.002824
878     9.374413
1280    9.191056
184     8.914492
Name: LotArea, dtype: float64

Log transformation correct: True



### 6.5 Creating Age-Based Features

New numerical features are created to represent the age and renovation history of a property at the time of sale.

Two features are introduced:
- HouseAge: Difference between the year of sale and construction year.
- YearsSinceRemodel: Difference between the year of sale and the most recent remodeling year.

These features will be generated within the preprocessing workflow to ensure consistency during training and inference.


In [67]:
age_features = [
    "YrSold",
    "YearBuilt",
    "YearRemodAdd"
]

print(X_train[age_features].isna().sum())

print("\nDescriptive statistics:")
display(X_train[age_features].describe())

print("\nPotential inconsistencies:")
print(
    "Sale before construction:",
    (X_train["YrSold"] < X_train["YearBuilt"]).sum()
)

print(
    "Sale before remodeling:",
    (X_train["YrSold"] < X_train["YearRemodAdd"]).sum()
)

YrSold          0
YearBuilt       0
YearRemodAdd    0
dtype: int64

Descriptive statistics:


,YrSold,YearBuilt,YearRemodAdd
count,1168.000000,1168.000000,1168.000000
mean,2007.807363,1970.830479,1984.824486
std,1.316840,30.090643,20.401140
min,2006.000000,1872.000000,1950.000000
25%,2007.000000,1954.000000,1967.000000
50%,2008.000000,1972.000000,1993.000000
75%,2009.000000,2000.000000,2003.000000
max,2010.000000,2010.000000,2010.000000



Potential inconsistencies:
Sale before construction: 0
Sale before remodeling: 1


In [68]:
inconsistent_remodel = X_train.loc[
    X_train["YearRemodAdd"] > X_train["YrSold"],
    ["YrSold", "YearBuilt", "YearRemodAdd", "SalePrice"]
    if "SalePrice" in X_train.columns
    else ["YrSold", "YearBuilt", "YearRemodAdd"]
].copy()

inconsistent_remodel["YearsSinceRemodel"] = (
    inconsistent_remodel["YrSold"]
    - inconsistent_remodel["YearRemodAdd"]
)

display(inconsistent_remodel)

,YrSold,YearBuilt,YearRemodAdd,YearsSinceRemodel
523,2007,2007,2008,-1


### 6.6 Custom Transformer for Age-Based Features

A custom scikit-learn transformer is implemented to generate two age-related features:

- HouseAge: Age of the property at the time of sale.
- YearsSinceRemodel: Number of years since the most recent remodeling, clipped at zero.

The transformer preserves the original input features and adds the newly engineered features.

It is designed to be integrated into the preprocessing pipeline.

In [69]:
class AgeFeaturesTransformer(TransformerMixin, BaseEstimator):

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object
        )
        self.n_features_names_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")

        X = X.copy()

        X["HouseAge"] = (X["YrSold"] - X["YearRemodAdd"]).clip(lower=0)
        X["YearsSinceRemodel"] = (X["YrSold"] - X["YearRemodAdd"]).clip(lower=0)

        return X

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")

        if input_features is None:
            input_features = self.feature_names_in_

        return np.asarray(
            [*input_features, "HouseAge", "YearsSinceRemodel"],
            dtype=object
        )

In [70]:
age_transformer = AgeFeaturesTransformer()

age_transformed = age_transformer.fit_transform(
    X_train[["YrSold", "YearBuilt", "YearRemodAdd"]]
)

print("Original shape:", X_train[
    ["YrSold", "YearBuilt", "YearRemodAdd"]
].shape)

print("Transformed shape:", age_transformed.shape)

display(age_transformed.head())

print("\nFeature names:")
print(age_transformer.get_feature_names_out())

Original shape: (1168, 3)
Transformed shape: (1168, 5)


,YrSold,YearBuilt,YearRemodAdd,HouseAge,YearsSinceRemodel
1356,2008,1966,1966,42,42
989,2006,2006,2006,0,0
878,2010,1961,1995,15,15
1280,2009,2002,2002,7,7
184,2006,1908,1991,15,15



Feature names:
['YrSold' 'YearBuilt' 'YearRemodAdd' 'HouseAge' 'YearsSinceRemodel']


### 6.7 Integrating Age Features into ColumnTransformer

The custom AgeFeaturesTransformer is integrated into the main preprocessing workflow.

It receives the original construction, remodeling, and sale year features, preserves them, and generates two additional age-based features.

The ColumnTransformer ensures that these input columns are not duplicated by the passthrough mechanism.

In [71]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            nominal_absence_pipeline,
            nominal_absence_features
        ),
        (
            "nominal_other",
            nominal_other_pipeline,
            nominal_other_features
        ),
        (
            "masonry_type",
            masonry_type_pipeline,
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "lot_frontage",
            lot_frontage_pipeline,
            ["LotFrontage"]
        ),
        (
            "garage_year",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "log_features",
            log_transformer,
            log_features_no_missing
        ),
        (
            "age_features",
            AgeFeaturesTransformer(),
            ["YrSold", "YearBuilt", "YearRemodAdd"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [72]:
X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

print("Transformed shape:", X_train_transformed.shape)

print("\nNew features:")
display(
    X_train_transformed[
        ["HouseAge", "YearsSinceRemodel"]
    ].head()
)

print(
    "\nDuplicated columns:",
    X_train_transformed.columns.duplicated().sum()
)

print(
    "Total missing values:",
    X_train_transformed.isna().sum().sum()
)

Transformed shape: (1168, 242)

New features:


,HouseAge,YearsSinceRemodel
1356,42.0,42.0
989,0.0,0.0
878,15.0,15.0
1280,7.0,7.0
184,15.0,15.0



Duplicated columns: 0
Total missing values: 0


### 6.8 Creating Total Property Area Feature

A new numerical feature, TotalSF, is proposed to represent the combined above-ground living area and basement area of a property.

TotalSF is calculated as the sum of GrLivArea and TotalBsmtSF.

The feature is intended to provide the model with a measure of total property area, while retaining the original area-related features.

In [73]:
area_features = [
    "GrLivArea",
    "TotalBsmtSF"
]

print("Missing values:")
print(X_train[area_features].isna().sum())

print("\nDescriptive statistics:")
display(X_train[area_features].describe())

print("\nProperties without a basement:")
print((X_train["TotalBsmtSF"] == 0).sum())

print("\nNegative area values:")
print((X_train[area_features] < 0).sum())

Missing values:
GrLivArea      0
TotalBsmtSF    0
dtype: int64

Descriptive statistics:


,GrLivArea,TotalBsmtSF
count,1168.000000,1168.000000
mean,1510.523116,1055.102740
std,519.955714,417.583282
min,334.000000,0.000000
25%,1120.750000,797.500000
50%,1461.500000,990.000000
75%,1788.000000,1297.250000
max,4676.000000,3206.000000



Properties without a basement:
29

Negative area values:
GrLivArea      0
TotalBsmtSF    0
dtype: int64



### 6.9 Custom Transformer for Area-Based Features

A custom transformer is implemented to calculate the TotalSF feature as the sum of above-ground living area and basement area.

The transformer preserves the original input features and adds TotalSF as an additional numerical feature.

It follows the scikit-learn transformer interface to support integration with ColumnTransformer.


In [74]:
# Unlike AgeFeaturesTransformer, this transformer returns only the newly
# engineered feature (TotalSF), without preserving the input columns.
#
# GrLivArea is already processed by the logarithmic transformation,
# so returning it again would create a duplicate feature.
#
# TotalBsmtSF is used to calculate TotalSF, but it must be preserved
# separately using a passthrough branch in ColumnTransformer.
#
# This design allows multiple transformers to use the same input columns
# while keeping the final feature set free of duplicated columns.
class AreaFeaturesTransformer(TransformerMixin, BaseEstimator):

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object
        )

        self.n_features_names_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")

        total_sf = (X["GrLivArea"] + X["TotalBsmtSF"])

        return pd.DataFrame(
            {"TotalSF": total_sf},
            index=X.index
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")

        return np.asarray(["TotalSF"], dtype=object)

In [75]:
area_transformer = AreaFeaturesTransformer()

area_transformed = area_transformer.fit_transform(
    X_train[["GrLivArea", "TotalBsmtSF"]]
)

print("Original shape:", X_train[
    ["GrLivArea", "TotalBsmtSF"]
].shape)

print("Transformed shape:", area_transformed.shape)

print("\nFeature names:")
print(area_transformer.get_feature_names_out())

display(area_transformed.head())

print(
    "\nTotalSF calculation correct:",
    np.allclose(
        area_transformed["TotalSF"],
        X_train["GrLivArea"] + X_train["TotalBsmtSF"]
    )
)

Original shape: (1168, 2)
Transformed shape: (1168, 1)

Feature names:
['TotalSF']


,TotalSF
1356,1756
989,2346
878,2264
1280,3146
184,1756



TotalSF calculation correct: True


In [76]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            nominal_absence_pipeline,
            nominal_absence_features
        ),
        (
            "nominal_other",
            nominal_other_pipeline,
            nominal_other_features
        ),
        (
            "masonry_type",
            masonry_type_pipeline,
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "lot_frontage",
            lot_frontage_pipeline,
            ["LotFrontage"]
        ),
        (
            "garage_year",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "log_features",
            log_transformer,
            log_features_no_missing
        ),
        (
            "age_features",
            AgeFeaturesTransformer(),
            ["YrSold", "YearBuilt", "YearRemodAdd"]
        ),
        (
            "area_features",
            AreaFeaturesTransformer(),
            ["GrLivArea", "TotalBsmtSF"]
        ),
        (
            "basement_area",
            "passthrough",
            ["TotalBsmtSF"]
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [77]:

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

print("Transformed shape:", X_train_transformed.shape)
print("Missing values:", X_train_transformed.isna().sum().sum())
print("Duplicated columns:", X_train_transformed.columns.duplicated().sum())

print("\nArea features:")
display(
    X_train_transformed[
        ["GrLivArea", "TotalBsmtSF", "TotalSF"]
    ].head()
)

print(
    "\nTotalSF correct:",
    np.allclose(
        X_train_transformed["TotalSF"],
        X_train["GrLivArea"] + X_train["TotalBsmtSF"]
    )
)

print(
    "GrLivArea log correct:",
    np.allclose(
        X_train_transformed["GrLivArea"],
        np.log1p(X_train["GrLivArea"])
    )
)


Transformed shape: (1168, 243)
Missing values: 0
Duplicated columns: 0

Area features:


,GrLivArea,TotalBsmtSF,TotalSF
1356,6.794587,864.0,1756.0
989,7.363280,770.0,2346.0
878,7.052721,1109.0,2264.0
1280,7.361375,1573.0,3146.0
184,7.133296,504.0,1756.0



TotalSF correct: True
GrLivArea log correct: True


### 6.10 Creating Total Bathroom Feature

A new numerical feature, TotalBathrooms, is proposed to combine information about full and half bathrooms across the property.

Full bathrooms receive a weight of 1.0, while half bathrooms receive a weight of 0.5.

The feature will be implemented within the preprocessing pipeline, while preserving the original bathroom-related variables.

In [78]:
bathroom_features = [
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath"
]

print("Missing values:")
print(X_train[bathroom_features].isna().sum())

print("\nDescriptive statistics:")
display(X_train[bathroom_features].describe())

print("\nNegative values:")
print((X_train[bathroom_features] < 0).sum())

print("\nNumber of unique values:")
print(X_train[bathroom_features].nunique())

Missing values:
FullBath        0
HalfBath        0
BsmtFullBath    0
BsmtHalfBath    0
dtype: int64

Descriptive statistics:


,FullBath,HalfBath,BsmtFullBath,BsmtHalfBath
count,1168.000000,1168.000000,1168.000000,1168.000000
mean,1.554795,0.379281,0.423801,0.060788
std,0.554249,0.502759,0.518070,0.246108
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000,0.000000
max,3.000000,2.000000,3.000000,2.000000



Negative values:
FullBath        0
HalfBath        0
BsmtFullBath    0
BsmtHalfBath    0
dtype: int64

Number of unique values:
FullBath        4
HalfBath        3
BsmtFullBath    4
BsmtHalfBath    3
dtype: int64


### 6.11 Custom Transformer for Bathroom Features

A custom scikit-learn transformer is implemented to calculate TotalBathrooms by combining full and half bathrooms.

Full bathrooms receive a weight of 1.0, while half bathrooms receive a weight of 0.5.

The transformer returns only the newly engineered feature, allowing the original bathroom-related features to be preserved separately in ColumnTransformer.

In [79]:
class BathroomFeaturesTransformer(TransformerMixin, BaseEstimator):

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(
            X.columns,
            dtype=object
        )

        self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")

        total_bathrooms = (
            X["FullBath"]
            + X["BsmtFullBath"]
            + 0.5 * (X["HalfBath"] + X["BsmtHalfBath"])
        )

        return pd.DataFrame(
            {"TotalBathrooms": total_bathrooms},
            index=X.index
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_in_")

        return np.array(["TotalBathrooms"], dtype=object)

In [80]:
bathroom_transformer = BathroomFeaturesTransformer()

bathroom_transformed = bathroom_transformer.fit_transform(
    X_train[bathroom_features]
)

print("Original shape:", X_train[bathroom_features].shape)
print("Transformed shape:", bathroom_transformed.shape)

print("\nFeature names:")
print(bathroom_transformer.get_feature_names_out())

display(bathroom_transformed.head())

print(
    "\nNegative values:",
    (bathroom_transformed["TotalBathrooms"] < 0).sum()
)

Original shape: (1168, 4)
Transformed shape: (1168, 1)

Feature names:
['TotalBathrooms']


,TotalBathrooms
1356,1.0
989,2.5
878,2.0
1280,3.0
184,1.0



Negative values: 0


### 6.12 Integrating Bathroom Features into ColumnTransformer

The BathroomFeaturesTransformer is integrated into the main preprocessing workflow to generate the TotalBathrooms feature.

The four original bathroom-related features are preserved using an explicit passthrough branch.

This approach retains both the individual bathroom counts and the newly engineered aggregate feature without creating duplicated columns.

In [81]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "ordinal_absence",
            ordinal_absence_pipeline,
            ordinal_absence_features
        ),
        (
            "ordinal_other",
            ordinal_other_pipeline,
            ordinal_other_features
        ),
        (
            "nominal_absence",
            nominal_absence_pipeline,
            nominal_absence_features
        ),
        (
            "nominal_other",
            nominal_other_pipeline,
            nominal_other_features
        ),
        (
            "masonry_type",
            masonry_type_pipeline,
            ["MasVnrType", "MasVnrArea"]
        ),
        (
            "lot_frontage",
            lot_frontage_pipeline,
            ["LotFrontage"]
        ),
        (
            "garage_year",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            ),
            ["GarageYrBlt"]
        ),
        (
            "masonry_area",
            PositiveMedianImputer(),
            ["MasVnrArea"]
        ),
        (
            "log_features",
            log_transformer,
            log_features_no_missing
        ),
        (
            "age_features",
            AgeFeaturesTransformer(),
            ["YrSold", "YearBuilt", "YearRemodAdd"]
        ),
        (
            "area_features",
            AreaFeaturesTransformer(),
            ["GrLivArea", "TotalBsmtSF"]
        ),
        (
            "basement_area",
            "passthrough",
            ["TotalBsmtSF"]
        ),
        (
            "bathroom_features",
            BathroomFeaturesTransformer(),
            bathroom_features
        ),
        (
            "bathroom_original",
            "passthrough",
            bathroom_features
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [82]:

X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed = pd.DataFrame(
    X_train_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)

print("Transformed shape:", X_train_transformed.shape)
print("Missing values:", X_train_transformed.isna().sum().sum())
print("Duplicated columns:", X_train_transformed.columns.duplicated().sum())

print("\nBathroom features:")
display(
    X_train_transformed[
        [
            "FullBath",
            "HalfBath",
            "BsmtFullBath",
            "BsmtHalfBath",
            "TotalBathrooms"
        ]
    ].head()
)

expected_bathrooms = (
    X_train["FullBath"]
    + X_train["BsmtFullBath"]
    + 0.5 * (
        X_train["HalfBath"]
        + X_train["BsmtHalfBath"]
    )
)

print(
    "\nTotalBathrooms correct:",
    np.allclose(
        X_train_transformed["TotalBathrooms"],
        expected_bathrooms
    )
)


Transformed shape: (1168, 244)
Missing values: 0
Duplicated columns: 0

Bathroom features:


,FullBath,HalfBath,BsmtFullBath,BsmtHalfBath,TotalBathrooms
1356,1.0,0.0,0.0,0.0,1.0
989,2.0,1.0,0.0,0.0,2.5
878,1.0,0.0,1.0,0.0,2.0
1280,2.0,0.0,1.0,0.0,3.0
184,1.0,0.0,0.0,0.0,1.0



TotalBathrooms correct: True



## 7. Final Preprocessing Validation

The complete preprocessing workflow is validated to ensure consistency between training data and unseen observations.

The preprocessor is fitted exclusively on the training dataset and then applied to the holdout dataset using transform().

The validation checks feature dimensionality, feature names, missing values, and numerical compatibility.

No model training or evaluation is performed at this stage.


In [83]:
# Fit preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the fitted preprocessor to holdout data
X_test_processed = preprocessor.transform(X_test)

# Retrieve feature names
feature_names = preprocessor.get_feature_names_out()

# Convert outputs into DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

print(
    "\nSame feature columns:",
    X_train_processed.columns.equals(X_test_processed.columns)
)

print(
    "Train missing values:",
    X_train_processed.isna().sum().sum()
)

print(
    "Test missing values:",
    X_test_processed.isna().sum().sum()
)

print(
    "Train infinite values:",
    np.isinf(X_train_processed.to_numpy(dtype=float)).sum()
)

print(
    "Test infinite values:",
    np.isinf(X_test_processed.to_numpy(dtype=float)).sum()
)

print(
    "Duplicated feature names:",
    X_train_processed.columns.duplicated().sum()
)

Train shape: (1168, 244)
Test shape: (292, 244)

Same feature columns: True
Train missing values: 0
Test missing values: 0
Train infinite values: 0
Test infinite values: 0
Duplicated feature names: 0



## 8. Feature Engineering Summary

The feature engineering workflow was implemented using scikit-learn Pipeline, ColumnTransformer, and custom transformers.

### Missing Value Imputation
- Median imputation for LotFrontage.
- Constant-value imputation for features where missing values represent absence.
- Most-frequent imputation for selected categorical features.
- Custom imputation for masonry-related features.

### Categorical Encoding
- OrdinalEncoder for categorical features with a meaningful order.
- OneHotEncoder for nominal categorical features.
- Handling of previously unseen categories.

### Numerical Transformations
- Logarithmic transformation of LotArea, LotFrontage, 1stFlrSF, and GrLivArea.
- Creation of four additional features:
  - HouseAge
  - YearsSinceRemodel
  - TotalSF
  - TotalBathrooms

### Final Validation
- Training observations: 1,168
- Holdout observations: 292
- Final feature count: 244
- Missing values: 0
- Infinite values: 0
- Duplicated feature names: 0
- Identical feature structure across training and holdout datasets.

### Next Steps

The preprocessing workflow is ready for integration with a machine learning model.

In the next stage, custom transformers will be extracted into reusable Python modules, and the full preprocessing workflow will be incorporated into a model Pipeline.

Model performance and the usefulness of engineered features will be evaluated using cross-validation.
